In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   TS-FORECASTING KAGGLE — v39  (feature_v lags + EMA multi-span)          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  v38 → v39: 2 thay đổi dựa trên phân tích OOF + feature importance        ║
║                                                                              ║
║  [1] THÊM feature_v VÀO LAG SOURCE COLS                                    ║
║      Bằng chứng: feature_v là top-2 cho h=10 và h=25 nhưng không có       ║
║      lag/rolling/DiffRoC features → đây là gap lớn nhất                   ║
║      Thêm: 5 lags + 6 rolling + 1 ewm + 2 diff + 1 rank + 5 DiffRoC      ║
║             + 3 EMA mới (ewm5, ewm25, crossover) = 23 features mới        ║
║                                                                              ║
║  [2] EMA MULTI-SPAN cho tất cả source cols                                 ║
║      Bằng chứng: temporal residual plot có oscillating pattern              ║
║      → cyclical signal chưa được capture đủ                               ║
║      ewm(span=5)  = short-term momentum                                    ║
║      ewm(span=25) = medium-term trend                                      ║
║      crossover    = ewm5 - ewm25 (momentum direction signal)               ║
║      Thêm: 3 features × 5 existing cols = 15 features mới                 ║
║                                                                              ║
║  Tổng: ~228 features (v38: 193, v37: 253)                                 ║
║  RAM ước tính: ~24-25 GB (v37: 28.3GB với 253 features + neutralization)  ║
║                                                                              ║
║  GIỮ NGUYÊN: VAL_THRESHOLD=3500, per-horizon, 2-stage, LGB params        ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import gc
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
T0 = time.time()

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — GIỮ NGUYÊN v36/v38
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH  = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

VAL_THRESHOLD  = 3500    # KHÔNG THAY ĐỔI — lý do score cao
HORIZONS       = [1, 3, 10, 25]
SEEDS          = [42, 2024, 12345]
CLIP_Q_LOW     = 0.005
CLIP_Q_HIGH    = 0.995

LGB_PARAMS = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.015,
    'n_estimators':      5000,
    'num_leaves':        90,
    'min_child_samples': 200,
    'feature_fraction':  0.65,
    'bagging_fraction':  0.75,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':        10.0,
    'verbosity':         -1,
    'n_jobs':            -1,
}

# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES — giữ nguyên
# ══════════════════════════════════════════════════════════════════════════════
def _clip01(x): return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred   = np.asarray(y_pred,   dtype=np.float64)
    w        = np.asarray(w,        dtype=np.float64)
    denom    = np.sum(w * y_target**2)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(1.0 - _clip01(np.sum(w*(y_target-y_pred)**2) / denom)))


def log_ram(label=''):
    try:
        import psutil
        mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
        print(f'   [RAM {label}]: {mb:,.0f} MB ({mb/1024:.1f} GB)')
    except ImportError:
        pass


# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════
def build_features_polars(train_path, test_path):
    print('⏳ Feature Engineering v39 (feature_v lags + EMA multi-span)...')

    # ── Đọc & concat ─────────────────────────────────────────────────────────
    df_train = pl.read_parquet(train_path)
    df_test  = pl.read_parquet(test_path)

    df_train = df_train.with_columns(pl.lit(1, dtype=pl.Int8).alias('is_train'))
    df_test  = df_test.with_columns(
        pl.lit(0, dtype=pl.Int8).alias('is_train'),
        pl.lit(None).cast(pl.Float64).alias('y_target'),
        pl.lit(None).cast(pl.Float64).alias('weight'),
    ).select(df_train.columns)

    df = pl.concat([df_train, df_test]).sort(
        ['code', 'sub_code', 'sub_category', 'horizon', 'ts_index'])
    del df_train, df_test
    gc.collect()
    print(f'   concat: {len(df):,} rows | {df.width} cols')
    log_ram('after concat')

    # ── Target Encoding — fit CHỈ trên train ≤ 3500, KHÔNG LEAK ──────────────
    fit_df      = df.filter((pl.col('is_train') == 1) &
                            (pl.col('ts_index') <= VAL_THRESHOLD))
    global_mean = float(fit_df.select(pl.col('y_target').mean()).item())

    cat_enc  = fit_df.group_by('sub_category').agg(
        pl.col('y_target').mean().alias('sub_category_enc'))
    code_enc = fit_df.group_by('sub_code').agg(
        pl.col('y_target').mean().alias('sub_code_enc'))

    df = (df.join(cat_enc, on='sub_category', how='left')
            .with_columns(pl.col('sub_category_enc').fill_null(global_mean)))
    df = (df.join(code_enc, on='sub_code', how='left')
            .with_columns(pl.col('sub_code_enc').fill_null(global_mean)))
    del fit_df, cat_enc, code_enc
    gc.collect()

    # ── Interactions & CS z-score ─────────────────────────────────────────────
    exprs = []
    if 'feature_al' in df.columns and 'feature_am' in df.columns:
        exprs += [
            (pl.col('feature_al') - pl.col('feature_am')).alias('d_al_am'),
            (pl.col('feature_al') / (pl.col('feature_am') + 1e-7)).alias('r_al_am'),
        ]
    if 'feature_cg' in df.columns and 'feature_by' in df.columns:
        exprs.append((pl.col('feature_cg') - pl.col('feature_by')).alias('d_cg_by'))
    if exprs:
        df = df.with_columns(exprs)
        exprs.clear()

    cs_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']
    for c in cs_cols:
        if c in df.columns:
            exprs.append(
                ((pl.col(c) - pl.col(c).mean().over('ts_index')) /
                 (pl.col(c).std().over('ts_index') + 1e-7)).alias(f'{c}_cs'))

    exprs += [
        (np.sin(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_sin'),
        (np.cos(2 * np.pi * pl.col('ts_index') / 100.0)).alias('t_cos'),
    ]
    df = df.with_columns(exprs)
    exprs.clear()

    # ── Lag + Rolling + DiffRoC + [NEW] EMA multi-span ───────────────────────
    # [NEW v39] Thêm feature_v vào source cols
    # Bằng chứng: top-2 feature cho h=10 và h=25, không có lag features hiện tại
    LAG_COLS = ['feature_al', 'feature_am', 'feature_cg',
                'feature_by', 'feature_s',
                'feature_v']   # <-- MỚI trong v39

    target_cols = [c for c in LAG_COLS if c in df.columns]
    group_cols  = ['code', 'sub_code', 'sub_category', 'horizon']

    for c in target_cols:
        # ── v36 features (giữ nguyên) ─────────────────────────────────────
        for lag in [1, 3, 5, 10, 25]:
            exprs.append(pl.col(c).shift(lag).over(group_cols).alias(f'{c}_lag{lag}'))
        for w in [5, 10, 20]:
            exprs.append(pl.col(c).rolling_mean(w, min_periods=1).over(group_cols)
                           .alias(f'{c}_roll_mean_{w}'))
            exprs.append(pl.col(c).rolling_std(w, min_periods=1).over(group_cols)
                           .alias(f'{c}_roll_std_{w}'))
        exprs.append(pl.col(c).ewm_mean(span=10, min_periods=1, ignore_nulls=True)
                       .over(group_cols).alias(f'{c}_ewm10'))
        exprs.append(pl.col(c).diff(1).over(group_cols).alias(f'{c}_diff1'))
        exprs.append((pl.col(c).rank() / pl.count()).over('ts_index')
                       .alias(f'{c}_rank'))

        # ── DiffRoC features (từ v38) ─────────────────────────────────────
        exprs.append(pl.col(c).diff(3).over(group_cols)
                       .cast(pl.Float32).alias(f'{c}_diff3'))
        exprs.append(pl.col(c).diff(5).over(group_cols)
                       .cast(pl.Float32).alias(f'{c}_diff5'))
        for k in [5, 10]:
            shifted = pl.col(c).shift(k).over(group_cols)
            roc_expr = ((pl.col(c) - shifted) / (shifted.abs() + 1e-7)).clip(-5.0, 5.0)
            exprs.append(roc_expr.cast(pl.Float32).alias(f'{c}_roc{k}'))
        accel_expr = (
            pl.col(c)
            - 2 * pl.col(c).shift(1).over(group_cols)
            + pl.col(c).shift(2).over(group_cols)
        ).cast(pl.Float32).alias(f'{c}_accel')
        exprs.append(accel_expr)

        # ── [NEW v39] EMA multi-span ──────────────────────────────────────
        # ewm(5): short-term momentum — capture rapid regime changes
        # ewm(25): medium-term trend — capture longer cycle
        # crossover = ewm5 - ewm25: direction of momentum shift
        #   > 0 → short-term above trend (bullish momentum)
        #   < 0 → short-term below trend (bearish momentum)
        ewm5_alias  = f'{c}_ewm5'
        ewm25_alias = f'{c}_ewm25'
        exprs.append(
            pl.col(c).ewm_mean(span=5, min_periods=1, ignore_nulls=True)
              .over(group_cols).cast(pl.Float32).alias(ewm5_alias))
        exprs.append(
            pl.col(c).ewm_mean(span=25, min_periods=1, ignore_nulls=True)
              .over(group_cols).cast(pl.Float32).alias(ewm25_alias))

    df = df.with_columns(exprs)
    exprs.clear()

    # Tính crossover sau khi ewm5 và ewm25 đã được tạo
    # (phải tính riêng vì cần tham chiếu đến 2 cols mới vừa tạo)
    crossover_exprs = [
        (pl.col(f'{c}_ewm5') - pl.col(f'{c}_ewm25'))
        .cast(pl.Float32).alias(f'{c}_ema_cross')
        for c in target_cols
        if f'{c}_ewm5' in df.columns and f'{c}_ewm25' in df.columns
    ]
    if crossover_exprs:
        df = df.with_columns(crossover_exprs)

    n_new_cols = len(target_cols) * (3 + (1 if 'feature_v' in target_cols else 0))
    print(f'   Lag+DiffRoC+EMA done: {len(target_cols)} source cols')
    print(f'   [NEW] EMA multi-span: ewm5, ewm25, ema_cross × {len(target_cols)} cols')

    # ── Ép Float32, fill NaN → 0 ─────────────────────────────────────────────
    df = df.with_columns(cs.float().cast(pl.Float32).fill_null(0.0))

    total_cols = df.width
    print(f'   Total: {total_cols} columns')
    log_ram('before to_pandas')

    # ── Tách train/test → Pandas, xóa Polars ngay ────────────────────────────
    df_train_pd = df.filter(pl.col('is_train') == 1).drop('is_train').to_pandas()
    df_test_pd  = (df.filter(pl.col('is_train') == 0)
                     .drop(['is_train', 'y_target', 'weight'])
                     .to_pandas())
    del df
    gc.collect()
    log_ram('FE done')

    return df_train_pd, df_test_pd


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN PER-HORIZON — 2-stage, giữ nguyên v36/v38
# ══════════════════════════════════════════════════════════════════════════════
def solve_horizon(horizon, train_df, test_df):
    t_h = time.time()
    print(f"\n{'='*70}\n 🚀 HORIZON {horizon}\n{'='*70}")

    tr = train_df[train_df['horizon'] == horizon].copy()
    te = test_df[test_df['horizon'] == horizon].copy()

    EXCL  = {'id', 'code', 'sub_code', 'sub_category',
             'horizon', 'ts_index', 'weight', 'y_target'}
    feats = [c for c in tr.columns if c not in EXCL]

    # In thống kê features mới
    n_ema      = sum(1 for c in feats if '_ewm5' in c or '_ewm25' in c or '_ema_cross' in c)
    n_diffroc  = sum(1 for c in feats if any(x in c for x in ['_diff3','_diff5','_roc','_accel']))
    n_fv       = sum(1 for c in feats if c.startswith('feature_v_'))
    print(f'Data: train={len(tr):,}, test={len(te):,} | '
          f'Features: {len(feats)} (EMA={n_ema}, DiffRoC={n_diffroc}, feat_v={n_fv})')

    fit_m = tr['ts_index'] <= VAL_THRESHOLD
    val_m = tr['ts_index'] >  VAL_THRESHOLD

    X_fit = tr.loc[fit_m, feats];   y_fit = tr.loc[fit_m, 'y_target']
    w_fit = tr.loc[fit_m, 'weight']
    X_val = tr.loc[val_m, feats];   y_val = tr.loc[val_m, 'y_target']
    w_val = tr.loc[val_m, 'weight']
    X_all = tr[feats];              y_all = tr['y_target']
    w_all = tr['weight']
    ts_val = tr.loc[val_m, 'ts_index'].values

    val_pred   = np.zeros(len(y_val), dtype=np.float64)
    test_pred  = np.zeros(len(te),    dtype=np.float64)
    best_iters = []
    fi_cum     = np.zeros(len(feats), dtype=np.float64)

    for i, seed in enumerate(SEEDS, 1):
        print(f'   -> Seed {i}/{len(SEEDS)} (seed={seed})...')

        # Stage 1: val model → best_iteration
        mdl_val = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        mdl_val.fit(
            X_fit, y_fit, sample_weight=w_fit,
            eval_set=[(X_val, y_val)], eval_sample_weight=[w_val],
            callbacks=[
                lgb.early_stopping(200, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        bi = max(int(mdl_val.best_iteration_ or LGB_PARAMS['n_estimators']), 20)
        best_iters.append(bi)
        val_pred += mdl_val.predict(X_val) / len(SEEDS)
        del mdl_val
        gc.collect()

        # Stage 2: full model → test predictions
        mdl_full = lgb.LGBMRegressor(
            **{**LGB_PARAMS, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(
            X_all, y_all, sample_weight=w_all,
            callbacks=[lgb.log_evaluation(period=-1)],
        )
        test_pred += mdl_full.predict(te[feats]) / len(SEEDS)
        fi_cum    += mdl_full.feature_importances_
        del mdl_full
        gc.collect()

    score      = weighted_rmse_score(y_val.values, val_pred, w_val.values)
    q_lo, q_hi = np.quantile(y_fit.values, [CLIP_Q_LOW, CLIP_Q_HIGH])
    test_clip  = np.clip(test_pred, q_lo, q_hi)

    print(f'💡 h={horizon} | score={score:.6f} | '
          f'avg_iter={np.mean(best_iters):.0f} | '
          f'elapsed={(time.time()-t_h)/60:.1f} min')

    fi_df = pd.DataFrame({'feature': feats, 'importance': fi_cum / len(SEEDS)})\
              .sort_values('importance', ascending=False).reset_index(drop=True)

    out = {
        'horizon':        horizon,
        'id_test':        te['id'].values,
        'test_pred_raw':  test_pred,
        'test_pred_clip': test_clip,
        'id_val':         tr.loc[val_m, 'id'].values,
        'y_val':          y_val.values,
        'w_val':          w_val.values,
        'val_pred':       val_pred,
        'ts_val':         ts_val,
        'score_local':    score,
        'best_iters':     best_iters,
        'feature_imp':    fi_df,
    }

    del tr, te, X_fit, y_fit, w_fit, X_val, y_val, w_val, X_all, y_all, w_all
    del val_pred, test_pred, test_clip, fi_cum, ts_val
    gc.collect()

    return out


# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def create_diagnostic_plots(all_outputs):
    print('\n📊 Generating diagnostic plots...')
    HC    = {1: '#2196F3', 3: '#4CAF50', 10: '#FF9800', 25: '#E91E63'}
    V38_SCORES = {1: 0.0725, 3: 0.1420, 10: 0.2282, 25: 0.2730}
    DARK  = '#111827'

    # ── Plot 1: Score comparison v38 vs v39 ───────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor(DARK)
    ax.set_facecolor('#1f2937')

    h_list = sorted([o['horizon'] for o in all_outputs])
    s_v39  = [next(o['score_local'] for o in all_outputs if o['horizon'] == h)
              for h in h_list]
    s_v38  = [V38_SCORES[h] for h in h_list]
    x      = np.arange(len(h_list))

    ax.bar(x - 0.2, s_v38, 0.35, label='v38 (Kaggle=0.2604)',
           color='#4B5563', edgecolor='#6B7280')
    ax.bar(x + 0.2, s_v39, 0.35, label='v39',
           color=[HC[h] for h in h_list], edgecolor='#9CA3AF')
    for i, (s38, s39) in enumerate(zip(s_v38, s_v39)):
        d    = s39 - s38
        sign = '+' if d >= 0 else ''
        ax.text(i + 0.2, s39 + 0.003, f'{sign}{d:.4f}',
                ha='center', va='bottom', fontsize=9,
                color='#4ade80' if d >= 0 else '#f87171')

    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}' for h in h_list], color='#D1D5DB')
    ax.tick_params(colors='#9CA3AF')
    ax.set_ylabel('Local Score', color='#D1D5DB')
    ax.set_title('Score per Horizon: v38 vs v39', color='#F9FAFB', pad=10)
    ax.legend(fontsize=9, facecolor='#374151', labelcolor='#D1D5DB',
               edgecolor='#4B5563')
    for sp in ax.spines.values():
        sp.set_edgecolor('#374151')
    plt.tight_layout()
    plt.savefig('diag1_scores.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag1_scores.png')

    # ── Plot 2: OOF Pred vs True scatter ──────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h  = out['horizon']
        ax.set_facecolor('#1f2937')
        yt = out['y_val']; yp = out['val_pred']
        n  = min(3000, len(yt))
        idx = np.random.choice(len(yt), n, replace=False)
        ax.scatter(yt[idx], yp[idx], s=4, alpha=0.3, color=HC[h])
        lo = min(yt.min(), yp.min())
        hi = max(yt.max(), yp.max())
        ax.plot([lo, hi], [lo, hi], 'w--', lw=1, alpha=0.5)
        ax.set_title(f'h={h}  score={out["score_local"]:.4f}',
                     color='#F9FAFB', fontsize=10)
        ax.set_xlabel('y_true', color='#9CA3AF', fontsize=8)
        ax.set_ylabel('y_pred', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')
    plt.suptitle('OOF Pred vs True', color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag2_pred_vs_true.png', dpi=130, bbox_inches='tight', facecolor=DARK)
    plt.close()
    print('   ✓ diag2_pred_vs_true.png')

    # ── Plot 3: Residual theo thời gian ───────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.patch.set_facecolor(DARK)
    for ax, out in zip(axes, sorted(all_outputs, key=lambda o: o['horizon'])):
        h   = out['horizon']
        ax.set_facecolor('#1f2937')
        ts  = out['ts_val']
        res = out['y_val'] - out['val_pred']
        bins = np.percentile(ts, np.linspace(0, 100, 51))
        bx, by, be = [], [], []
        for j in range(len(bins)-1):
            m = (ts >= bins[j]) & (ts < bins[j+1])
            if m.sum() > 20:
                bx.append((bins[j]+bins[j+1])/2)
                by.append(np.mean(res[m]))
                be.append(np.std(res[m]) / np.sqrt(m.sum()))
        bx, by, be = map(np.array, [bx, by, be])
        ax.plot(bx, by, color=HC[h], lw=1.5)
        ax.fill_between(bx, by - be, by + be, alpha=0.2, color=HC[h])
        ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.5)
        ax.set_title(f'h={h}  temporal residual', color='#F9FAFB', fontsize=10)
        ax.set_xlabel('ts_index', color='#9CA3AF', fontsize=8)
        ax.set_ylabel('mean residual', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')
    plt.suptitle('Residual theo thời gian — Phẳng = tốt | Spike = regime shift',
                 color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag3_temporal_residual.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag3_temporal_residual.png')

    # ── Plot 4: Feature importance top 20 với màu mới ─────────────────────────
    # Màu: vàng cam = EMA mới, xanh nhạt = DiffRoC (v38), màu horizon = v36
    # Màu tím = feature_v lags (mới nhất)
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.patch.set_facecolor(DARK)
    from matplotlib.patches import Patch

    for ax, out in zip(axes.flatten(), sorted(all_outputs, key=lambda o: o['horizon'])):
        h   = out['horizon']
        fi  = out['feature_imp'].head(20)
        ax.set_facecolor('#1f2937')

        colors = []
        for feat in fi['feature']:
            if any(x in feat for x in ['_ewm5', '_ewm25', '_ema_cross']):
                colors.append('#f59e0b')   # amber = EMA mới
            elif feat.startswith('feature_v_'):
                colors.append('#a855f7')   # purple = feature_v lag (mới)
            elif any(x in feat for x in ['_diff3','_diff5','_roc','_accel']):
                colors.append('#22d3ee')   # cyan = DiffRoC (v38)
            else:
                colors.append(HC[h])       # horizon color = v36 feature

        ax.barh(range(len(fi)), fi['importance'], color=colors, edgecolor='#374151')
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi['feature'], fontsize=7, color='#D1D5DB')
        ax.invert_yaxis()
        ax.set_title(f'h={h}  Feature Importance Top 20', color='#F9FAFB', fontsize=11)
        ax.set_xlabel('importance', color='#9CA3AF', fontsize=8)
        ax.tick_params(colors='#6B7280', labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor('#374151')

        legend_elems = [
            Patch(facecolor='#f59e0b', label='EMA multi-span (v39 new)'),
            Patch(facecolor='#a855f7', label='feature_v lags (v39 new)'),
            Patch(facecolor='#22d3ee', label='DiffRoC (v38)'),
            Patch(facecolor=HC[h],     label='v36 feature'),
        ]
        ax.legend(handles=legend_elems, fontsize=7.5, facecolor='#374151',
                  labelcolor='#D1D5DB', edgecolor='#4B5563', loc='lower right')

    plt.suptitle(
        'Feature Importance v39\n'
        'Vàng=EMA mới | Tím=feat_v lags | Cyan=DiffRoC | Màu=v36',
        color='#D1D5DB', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('diag4_feature_importance.png', dpi=130, bbox_inches='tight',
                facecolor=DARK)
    plt.close()
    print('   ✓ diag4_feature_importance.png')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':

    df_train_all, df_test_all = build_features_polars(TRAIN_PATH, TEST_PATH)
    log_ram('after FE')

    all_outputs = []
    for h in HORIZONS:
        all_outputs.append(solve_horizon(h, df_train_all, df_test_all))

    # Tổng hợp
    sub_raw_parts, sub_clip_parts, oof_parts = [], [], []
    for out in all_outputs:
        sub_raw_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_raw']}))
        sub_clip_parts.append(
            pd.DataFrame({'id': out['id_test'], 'prediction': out['test_pred_clip']}))
        oof_parts.append(pd.DataFrame({
            'id': out['id_val'], 'y_true': out['y_val'],
            'y_pred': out['val_pred'], 'w': out['w_val'],
            'horizon': out['horizon'],
        }))

    sub_raw  = pd.concat(sub_raw_parts,  ignore_index=True)
    sub_clip = pd.concat(sub_clip_parts, ignore_index=True)
    oof      = pd.concat(oof_parts,      ignore_index=True)
    del sub_raw_parts, sub_clip_parts, oof_parts
    gc.collect()

    test_order = pd.read_parquet(TEST_PATH, columns=['id'])
    sub_clip   = test_order.merge(sub_clip, on='id', how='left').fillna(0)
    sub_raw    = test_order.merge(sub_raw,  on='id', how='left').fillna(0)
    del test_order

    agg_local = weighted_rmse_score(
        oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

    # Diagnostic plots
    create_diagnostic_plots(all_outputs)

    # Lưu files
    sub_clip.to_csv('submission.csv',        index=False)
    sub_raw.to_csv( 'submission_noclip.csv', index=False)
    oof.to_csv(     'oof_v39.csv',           index=False)

    # ── Final report ─────────────────────────────────────────────────────────
    V38_REF = {1: 0.0725, 3: 0.1420, 10: 0.2282, 25: 0.2730}
    print(f"\n{'='*70}")
    print(f'🏆 TỔNG KẾT v39 (feat_v lags + EMA multi-span)')
    print(f'   ref: v36 Kaggle=0.2554 | v38 Kaggle=0.2604')
    print(f"{'─'*70}")
    print(f'  {"h":>4} | {"v38":>8} | {"v39":>8} | {"delta":>8} | avg_iter')
    print(f'  {"─"*4}-+-{"─"*8}-+-{"─"*8}-+-{"─"*8}-+-{"─"*8}')
    for out in sorted(all_outputs, key=lambda o: o['horizon']):
        h   = out['horizon']
        s39 = out['score_local']
        s38 = V38_REF[h]
        d   = s39 - s38
        tag = '↑' if d > 0 else '↓'
        avg_i = np.mean(out['best_iters'])
        print(f'  h={h:>2} | {s38:>8.4f} | {s39:>8.4f} | {tag}{abs(d):.4f} | {avg_i:.0f}')
    print(f"{'─'*70}")
    print(f'🔥 Aggregate Local Score : {agg_local:.6f}')
    print(f"⏱️  Tổng thời gian : {(time.time()-T0)/60:.1f} phút")
    print(f"{'='*70}")

    print("""
CHECKLIST v39:

  [feature_v LAGS CÓ HIỆU QUẢ?]
  → diag4: bars tím (feature_v_*) trong top 10 của h=10 và h=25?
       Có → feature_v lags bổ sung thêm temporal signal quan trọng
       Không → feature_v là cross-sectional feature, lag không giúp

  [EMA MULTI-SPAN CÓ HIỆU QUẢ?]
  → diag4: bars vàng (_ewm5, _ewm25, _ema_cross) trong top 10?
  → diag3: temporal residual phẳng hơn so với v38?
       Có → EMA capture được cycle pattern trong data
       Không → cycle pattern không phải do momentum shift

  [BƯỚC TIẾP THEO NẾU SCORE TĂNG]
  → Thêm feature_bg vào lag source cols (top-3 h=25, chưa có lag)
  → Thêm TE granular: (sub_category × horizon) — đã có trong v37
     nhưng bị remove khi revert về v36, cần thêm lại cẩn thận

  [NẾU LOCAL TĂNG NHƯNG KAGGLE KHÔNG TĂNG]
  → EMA/feature_v lag overfit val region (ts 3500-3600)
  → Giảm lag windows hoặc giảm span của EMA
""")

In [ ]:
import os
print('Output files:')
for f in ['submission.csv','submission_noclip.csv','oof_v39.csv',
          'diag1_scores.png','diag2_pred_vs_true.png',
          'diag3_temporal_residual.png','diag4_feature_importance.png']:
    kb = os.path.getsize(f)//1024 if os.path.exists(f) else 0
    print(f'  {f}: {kb} KB' if kb else f'  {f}: MISSING')
print('\n✅ submission.csv sẵn sàng nộp!')